### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="phiusil_phishing",
    dataset_year="2023",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset",
    download_description="""

    mkdir -p local-data-warehouse/phiusil_phishing 
    wget -P local-data-warehouse/phiusil_phishing/ https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip
    unzip local-data-warehouse/phiusil_phishing/phiusiil+phishing+url+dataset.zip -d local-data-warehouse/phiusil_phishing/ 
    rm local-data-warehouse/phiusil_phishing/phiusiil+phishing+url+dataset.zip
""",
    # References
    academic_reference_bibtex="""@article{prasad2024phiusiil,
  title={PhiUSIIL: A diverse security profile empowered phishing URL detection framework based on similarity index and incremental learning},
  author={Prasad, Arvind and Chandra, Shalini},
  journal={Computers \\& Security},
  volume={136},
  pages={103545},
  year={2024},
  publisher={Elsevier}
}

""",
    academic_reference_bibtex_key="prasad2024phiusiil",
    license="CC BY 4.0",
    data_tags=["IID", "ForcedIIDFromTemporal"],
    curation_comments="""
    - We remove some duplicated URLs as they only differ in the URL length by one character, likely stemming from a data collection error.
    - We drop FILE_NAME as it is an identifier
    - 93.5% of the domains are unique. of the non-unique domains, only 0.0026 non-unique domains are phishing websites. Because domains like google.docs are so frequent and simple to classify, we never allow them in the test data and define a custom split.
    - We keep all domains that appear more than once as training data and do a random split for the remaining data. 
    - We don't use the domain column, to focus on generalization based on the provided features instead of memorization of domains.
    - The focus on unseen domains in the test data shifts the task slightly, but also mitigates the fact that we don't have temporal information although this is a temporal task.
    - We rename the labels to "Legitimate" and "Phishing" for better interpretability.
    """,
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="label",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="label",
    # group_on="Domain",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "PhiUSIIL_Phishing_URL_Dataset.csv")

# Drop duplicated URLs that only differ in the URL length by one character, likely stemming from a data collection error.
df = df.loc[df.URL.drop_duplicates().index]
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

predefined_train_indices = df.index[df.Domain.map(df.Domain.value_counts()>1)].tolist()

# Drop redundant features
df = df.drop(columns=["FILENAME", "URL", "Domain"])

df.label = df.label.map({0: "Phishing", 1: "Legitimate"})

print("Loaded data shape:", df.shape)

In [ ]:
# print(f"{df.Domain.nunique()/df.shape[0]:.4f} domains are unique")
# print(f"Only {df.loc[df.Domain.map(df.Domain.value_counts()>1),"label"].mean():.4f} non-unique domains are phishing websites")

In [ ]:
# # Domains that appear more than once and have fraudulent URLs
# vc = df.Domain.value_counts()
# for domain in vc[vc>1].index:
#     mean_label = df.loc[df.Domain==domain,"label"].mean()
#     if mean_label > 0:
#         print(domain, mean_label)


In [ ]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

# Move samples with non-unique domains to the training set and do a random split for the remaining data, ensuring that all samples from the same domain are in the same split.
for rep in splits:
    for fld in splits[rep]:
        train_idx, test_idx = splits[rep][fld]
        test_idx = sorted(list(set(test_idx)-set(predefined_train_indices)))
        splits[rep][fld] = (sorted(set(train_idx).union(predefined_train_indices)), test_idx)
        print(f"Repeat {rep}, Fold {fld}: Train samples: {len(splits[rep][fld][0])}, Test samples: {len(splits[rep][fld][1])}")


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)